In [18]:
import pandas as pd
import os

pd.set_option("display.max_rows", None)      # 모든 행 표시
pd.set_option("display.max_columns", None)   # 모든 열 표시
pd.set_option("display.width", None)         # 출력 폭 제한 해제
pd.set_option("display.max_colwidth", None)  # 셀 내용 생략 방지


folder_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\Test\test"
mer_list = sorted([file for file in os.listdir(folder_path) if file.endswith(".mer")])
for idx, start in enumerate(range(0, len(mer_list))):
    file_path = os.path.join(folder_path, mer_list[idx])
    print("작업파일 :", file_path)
    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        lines = file.readlines()

    data_start_idx = None
    for j, line in enumerate(lines):
        if "Measurem." in line:
            data_start_idx = j
            break
    # 데이터프레임 생성
    if data_start_idx is not None:

        # 컬럼명 추출 및 공백 제거
        columns = [col.strip() for col in lines[data_start_idx].strip().split(";")]

        # 데이터 부분 추출 및 가공
        data_lines = lines[data_start_idx + 1:]  # 컬럼명 제외, 데이터 부분
        data = [line.strip().split(";") for line in data_lines if line.strip()]

        # 데이터프레임 생성
        df = pd.DataFrame(data, columns=columns)

        # 컬럼 내부 데이터 정수형 변환
        df = df.apply(pd.to_numeric, errors="coerce")
        df = df[df["t(Entry)"] != -1.00].reset_index(drop=True)
        df = df[df["Measurem."].isin([10026, 20026])]

        df["TimeGroup"] = ((df["t(Entry)"] // 60) * 60).astype(int)
        display("df : ", df[(df["t(Entry)"] >= 5220) & (df["t(Entry)"] < 5280)])


        df["TimeGroup"] = (df["TimeGroup"].astype(str) + "~" +    (df["TimeGroup"] + 60).astype(str))
        df["New_Measurement"] = df["Measurem."] % 1000


        speed_mean_df = (
            df.groupby(["TimeGroup", "New_Measurement"])
              .agg(V_mean=("v[km/h]", "mean"), V_count=("v[km/h]", "count"))
              .reset_index()
        )
        #display("speed_mean_df : ", speed_mean_df)

        grouped = (
            df.groupby("TimeGroup")["v[km/h]"]
            .mean()
            .reset_index()
        )
        #display("grouped : ", grouped)



작업파일 : E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\Test\test\화성~서울(140-진입부22)_260426_001.mer


'df : '

,Measurem.,t(Entry),t(Exit),VehNo,Vehicle type,Line,v[km/h],b[m/s2],Occ,Pers,tQueue,VehLength[m],,TimeGroup
302641,20026,5222.57,-1.0,1885,100,0,18.8,3.04,0.03,1,1.7,3.75,NaN,5220
302858,20026,5224.72,-1.0,1886,100,0,20.8,1.64,0.08,1,0.0,3.75,NaN,5220
303047,20026,5226.95,-1.0,1887,100,0,28.3,0.94,0.05,1,0.0,4.76,NaN,5220
303281,20026,5229.57,-1.0,1888,100,0,20.2,-0.01,0.03,1,0.0,4.61,NaN,5220
303581,20026,5232.51,-1.0,1890,100,0,24.5,-0.04,0.09,1,0.0,4.21,NaN,5220
303694,20026,5234.13,-1.0,1895,100,0,33.5,-0.93,0.07,1,0.0,4.64,NaN,5220
303826,20026,5235.48,-1.0,1892,100,0,33.1,-0.97,0.02,1,0.0,4.61,NaN,5220
303954,10026,5236.72,-1.0,1891,100,0,3.6,-0.16,0.08,1,1.2,4.21,NaN,5220
304028,20026,5237.67,-1.0,1893,100,0,29.5,-0.01,0.03,1,0.0,4.61,NaN,5220
304252,20026,5240.05,-1.0,1896,100,0,40.7,-2.10,0.05,1,0.0,4.64,NaN,5220
